# Módulo 0 — Datos climáticos históricos: Jocolí / Lavalle, Mendoza

**Fuente:** Open-Meteo Archive API  
**Período:** 1990–2024 (35 años)  
**Variables:** temperatura 2m · precipitación · evapotranspiración de referencia (ET₀)

In [1]:
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO_URL = "https://github.com/Emilialandgrebe/tesis-maestria.git"
    ROOT = Path("/content/tesis-maestria")
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt"), "-q"],
        check=True,
    )
else:
    ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Entorno : {'Google Colab' if EN_COLAB else 'local'}")
print(f"ROOT    : {ROOT}")

Entorno : Google Colab
ROOT    : /content/tesis-maestria


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data.climate_fetcher import fetch_climate_data
from src.data.climate_features import (
    calcular_calor_verano,
    calcular_deficit_hidrico,
    calcular_heladas_tardias,
    calcular_horas_frio,
)

np.random.seed(42)

df = fetch_climate_data()

horas_frio      = calcular_horas_frio(df)
heladas         = calcular_heladas_tardias(df)
calor_verano    = calcular_calor_verano(df)
deficit_hidrico = calcular_deficit_hidrico(df)

In [ ]:
print("=" * 60)
print("RANGO DE FECHAS")
print("=" * 60)
print(f"Inicio : {df.index.min()}")
print(f"Fin    : {df.index.max()}")
print(f"Total  : {len(df):,} registros horarios")
print(f"         ({len(df) / 8760:.1f} años aprox.)")

In [ ]:
print("PRIMERAS FILAS")
df.head()

In [ ]:
print("ÚLTIMAS FILAS")
df.tail()

In [ ]:
print("ESTRUCTURA DEL DATAFRAME")
df.info()

In [ ]:
print("ESTADÍSTICAS DESCRIPTIVAS")
df.describe()

In [ ]:
print("=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(3)
resumen_nulos = nulos.to_frame("cantidad").assign(porcentaje=pct)
print(resumen_nulos)
print()
if nulos.sum() == 0:
    print("Sin valores nulos. Dataset completo.")
else:
    print(f"ATENCIÓN: {nulos.sum()} valores nulos en total.")

## Temperatura máxima media en verano (ene–feb)

In [ ]:
años = calor_verano.index
tendencia = np.polyfit(años, calor_verano.values, 1)
linea_tendencia = np.poly1d(tendencia)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(años, calor_verano.values, color="#e07b39", linewidth=1.8,
        marker="o", markersize=4, label="Tmax diaria media")

ax.fill_between(años, 35, 38, alpha=0.15, color="green",
                label="Rango óptimo pistacho (35–38 °C)")
ax.axhline(35, color="green", linewidth=0.8, linestyle="--")
ax.axhline(38, color="green", linewidth=0.8, linestyle="--")

ax.plot(años, linea_tendencia(años), color="#c0392b", linewidth=1.5,
        linestyle="-.", label=f"Tendencia ({tendencia[0]:+.3f} °C/año)")

ax.set_title("Temperatura máxima diaria media — enero y febrero\nJocolí, Lavalle, Mendoza (1990–2024)",
             fontsize=13)
ax.set_xlabel("Año")
ax.set_ylabel("°C")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Déficit hídrico anual (mm)

In [ ]:
años_dh = deficit_hidrico.index
tendencia_dh = np.polyfit(años_dh, deficit_hidrico.values, 1)
linea_tendencia_dh = np.poly1d(tendencia_dh)

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(años_dh, deficit_hidrico.values, color="#5b8db8", alpha=0.75,
       label="Deficit hídrico (mm)")

ax.axhline(600, color="#e07b39", linewidth=2, linestyle="--",
           label="600 mm (~6.000 m\u00b3/ha, mínimo plan de negocios)")

ax.plot(años_dh, linea_tendencia_dh(años_dh), color="#c0392b", linewidth=1.5,
        linestyle="-.", label=f"Tendencia ({tendencia_dh[0]:+.1f} mm/año)")

ax.set_title("Déficit hídrico anual (ET\u2080 - precipitación)\nJocolí, Lavalle, Mendoza (1990–2024)",
             fontsize=13)
ax.set_xlabel("Año")
ax.set_ylabel("mm / año")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Parámetros calibrados — exportar al Módulo 1

In [ ]:
from scipy import stats

UMBRAL_FRIO_CRITICO = 800

_dist, _params = "norm", stats.norm.fit(horas_frio.values)

PARAMS_CLIMA_JOCOLI = {
    "horas_frio_media":          float(horas_frio.mean()),
    "horas_frio_std":            float(horas_frio.std()),
    "horas_frio_dist":           _dist,
    "horas_frio_dist_params":    _params,
    "prob_anio_critico":         float((horas_frio < UMBRAL_FRIO_CRITICO).mean()),
    "prob_helada_tardia":        float((heladas > 0).mean()),
    "calor_verano_media":        float(calor_verano.mean()),
    "deficit_hidrico_media_mm":  float(deficit_hidrico.mean()),
}

print("PARAMS_CLIMA_JOCOLI = {")
for k, v in PARAMS_CLIMA_JOCOLI.items():
    print(f"    {k!r}: {v},")
print("}")